# M04C · Seaborn & Publication-Ready Styling
**Programming for Computational Biology (Python)** · Author: Prateek Paul · Last updated: 2026-10-29

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/meprateekpaul/ProgrammingCB/blob/main/modules/M04_Visualisation/M04C_Seaborn_and_Styling.ipynb)

⏱ ~90 min · Difficulty: ★★☆ · Prerequisites: M01A (Core Python), M04A (Biological Plots Theory), M04B (Matplotlib)


## 1. Learning Outcomes

By the end of this notebook, you will be able to:

- **Reshape**: Convert wide biological matrices into long (tidy) formats using `pandas.melt()` to interface seamlessly with Seaborn's statistical mapping.
- **Execute**: Produce high-level statistical plots: `barplot`, `histplot`/`kdeplot`, `boxplot`, `violinplot`, `heatmap`, and `clustermap`.
- **Facet**: Deconstruct high-dimensional multi-condition experiments across multi-panel grids using `hue`, `col`, and `row` with `FacetGrid` and `catplot`.
- **Calibrate**: Select and apply qualitative, sequential, and diverging color palettes compliant with colorblind accessibility guidelines.
- **Publish**: Construct complex showcase figures including an annotated Volcano plot and a hierarchically clustered expression heatmap.

## 2. Prerequisites


Before starting this notebook, you should be familiar with:
- Matplotlib's Figure and Axes object-oriented interface ([`M04B`](file:///modules/M04_Visualisation/M04B_Matplotlib.ipynb)).
- Reading CSV data into pandas DataFrames and filtering columns.
- The biological rationale behind volcano plots and heatmaps ([`M04A`](file:///modules/M04_Visualisation/M04A_Biological_Plots_Theory.ipynb)).


In [ ]:
# --- Setup -------------------------------------------------------
import sys, os
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !pip -q install biopython seaborn requests beautifulsoup4 lxml python-dotenv
    !git clone -q https://github.com/meprateekpaul/ProgrammingCB.git
    ROOT = Path("ProgrammingCB")
else:
    # Resolve repository root robustly from modules/ or repo root
    p = Path.cwd()
    while p != p.parent and not (p / "COURSE_PLAN.md").exists():
        p = p.parent
    ROOT = p if (p / "COURSE_PLAN.md").exists() else Path.cwd().resolve()

DATA_DIR = ROOT / "data"
OUT_DIR  = ROOT / "outputs" / "M04C"
OUT_DIR.mkdir(parents=True, exist_ok=True)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.ticker as ticker

print(f"Python: {sys.version.split()[0]}")
print(f"Working Directory: {Path.cwd()}")
print(f"DATA_DIR: {DATA_DIR} (exists: {DATA_DIR.exists()})")
print(f"OUT_DIR:  {OUT_DIR} (ready)")

## 4C.1 Why Seaborn & Tidy (Long) Data

### 4.1.1 Theory & Biological Context

Seaborn is built on top of Matplotlib and integrates tightly with `pandas` DataFrames. Unlike Matplotlib—which requires manually looping over categories, computing mean/error coordinates, and configuring colors—Seaborn operates declaratively on **Tidy Data** (long format).

### Tidy Data Principles (Hadley Wickham):
1. Each **variable** forms a column.
2. Each **observation** forms a row.
3. Each type of **observational unit** forms a table.

### Wide vs Long Format in Computational Biology:
- **Wide Format** (typical raw assay output):
  - Row = Gene; Columns = `Control_1`, `Control_2`, `Treated_1`, `Treated_2`.
  - While compact, you cannot easily map 'Condition' or 'Replicate' to aesthetics like `hue` or `style`.
- **Long / Tidy Format** (Seaborn requirement):
  - Columns = `['gene', 'sample', 'expression', 'condition']`.
  - Transformed using `pandas.melt()`.

### 4.1.2 Worked Example

In [ ]:
# 4C.1 Worked Example: Reshaping Wide Expression Matrix to Tidy Format
exp_path = DATA_DIR / "tables" / "expression_matrix.csv"
df_wide = pd.read_csv(exp_path)

print("Wide Format shape:", df_wide.shape)
print(df_wide.head(3))

# Melt the wide matrix into tidy format
sample_cols = [c for c in df_wide.columns if c != "gene"]
df_tidy = pd.melt(
    df_wide,
    id_vars=["gene"],
    value_vars=sample_cols,
    var_name="sample",
    value_name="expression"
)

# Extract experimental condition (Control vs Treated) from sample name
df_tidy["condition"] = df_tidy["sample"].apply(lambda s: "Control" if "Control" in s else "Treated")

print("Tidy Format shape:", df_tidy.shape)
print(df_tidy.head(6))

### 4.1.3 ✏️ Try It: Expression Table Reshaper

Write a function `melt_expression_table(df: pd.DataFrame) -> pd.DataFrame`:
- Melt `df` keeping `'gene'` as identifier, with variable column named `'sample'` and value column named `'expression'`.
- Add a column `'replicate'` that extracts the trailing number (e.g., `'Control_1'` -> `1`, `'Treated_3'` -> `3`) as an `int`.
- Return the tidy DataFrame.

In [ ]:
def melt_expression_table(df: pd.DataFrame) -> pd.DataFrame:
    # TODO: your code here
    ...

In [ ]:
# ✅ Test — run me. No output means you passed.
df_melted = melt_expression_table(df_wide)
assert "gene" in df_melted.columns and "expression" in df_melted.columns
assert "replicate" in df_melted.columns
assert df_melted["replicate"].dtype in [int, np.int64, np.int32]
print("Section 4C.1 passed ✔")

### 4.1.4 ⚠️ Common Mistakes

- **Passing wide matrices to sns.boxplot without melting**: Passing wide data causes each column to be plotted in isolation, preventing multi-factor grouping by condition or genotype.
- **Losing index metadata during melt**: Always ensure all identifier variables (e.g. `gene`, `chromosome`) are included in `id_vars`.

## 4C.2 The Core Biological Plots in Seaborn

### 4.2.1 Theory & Biological Context

Seaborn provides specialized statistical functions that automatically compute descriptive statistics, error intervals, and aesthetic encodings:

1. **`sns.barplot(data, x, y, hue)`**: Calculates mean (or custom estimator) with bootstrapping confidence intervals (`errorbar='se'` or `'ci'`).
2. **`sns.histplot(data, x, kde=True)`**: Histogram with Gaussian Kernel Density Estimate overlay.
3. **`sns.boxplot(data, x, y, hue)`**: Box-and-whisker plot with automatic categorical axis layout.
4. **`sns.violinplot(data, x, y, hue, split=True)`**: Shows density distributions; `split=True` creates split half-violins comparing two conditions directly inside one body!
5. **`sns.heatmap(data, annot=True, cmap=...)`**: High-level matrix heatmap.
6. **`sns.clustermap(data, z_score=0)`**: Hierarchical agglomerative clustering on rows and columns with dendrograms.

*(Note on Pie Charts: Seaborn intentionally does NOT include a pie chart function because empirical visualization research has proved that humans misjudge slice angles and areas far more than positions along an aligned scale).*

### 4.2.2 Worked Example

In [ ]:
# 4C.2 Worked Example: Six Core Seaborn Visualizations
sns.set_theme(style="ticks", font_scale=0.9)

# 1. Barplot with error bars
fig, axes = plt.subplots(2, 2, figsize=(11, 8), dpi=150)

# Panel 0,0: Barplot of Tissue Expression
tissues_df = pd.read_csv(DATA_DIR / "tables" / "expression_tissues.csv")
sns.barplot(data=tissues_df, x="tissue", y="tpm", ax=axes[0, 0], 
            palette="Blues_d", errorbar="se", capsize=0.1)
axes[0, 0].set_title("A. Mean Tissue Expression (±SE)", fontweight="bold")
axes[0, 0].tick_params(axis="x", rotation=30)
sns.despine(ax=axes[0, 0])

# Panel 0,1: Histplot + KDE
sns.histplot(data=tissues_df, x="tpm", kde=True, ax=axes[0, 1], 
             color="#2b8cbe", bins=12)
axes[0, 1].set_title("B. Expression Distribution (Histogram + KDE)", fontweight="bold")
sns.despine(ax=axes[0, 1])

# Panel 1,0: Boxplot
sns.boxplot(data=tissues_df, x="tissue", y="tpm", ax=axes[1, 0], 
            palette="Set2", width=0.5)
axes[1, 0].set_title("C. Tissue Expression Boxplot (IQR & Median)", fontweight="bold")
axes[1, 0].tick_params(axis="x", rotation=30)
sns.despine(ax=axes[1, 0])

# Panel 1,1: Violinplot
sns.violinplot(data=tissues_df, x="tissue", y="tpm", ax=axes[1, 1], 
               palette="Pastel1", inner="quartile")
axes[1, 1].set_title("D. Tissue Expression Violinplot", fontweight="bold")
axes[1, 1].tick_params(axis="x", rotation=30)
sns.despine(ax=axes[1, 1])

plt.tight_layout()
fig.savefig(OUT_DIR / "4C2_core_plots.png", dpi=300)
plt.show()

### 4.2.3 ✏️ Try It: Split Violin Plot Generator

Write a function `generate_split_violin(df_tidy: pd.DataFrame, top_genes: list[str]) -> plt.Axes`:
- Filter `df_tidy` for rows where `gene` is in `top_genes`.
- Plot a split violin plot: `x="gene"`, `y="expression"`, `hue="condition"`, `split=True`.
- Set `palette="muted"`, `inner="quartile"`.
- Set title: `"Comparative Expression (Control vs Treated)"`.
- Despine top and right axes.
- Return the `ax`.

In [ ]:
def generate_split_violin(df_tidy: pd.DataFrame, top_genes: list[str]) -> plt.Axes:
    # TODO: your code here
    ...

In [ ]:
# ✅ Test — run me. No output means you passed.
ax_violin = generate_split_violin(df_tidy, ["TP53", "BRCA1"])
assert ax_violin.get_title() == "Comparative Expression (Control vs Treated)"
plt.close(ax_violin.figure)
print("Section 4C.2 passed ✔")

### 4.2.4 ⚠️ Common Mistakes

- **Default 95% CI error bar interpretation**: By default in Seaborn, error bars on `barplot` represent 95% bootstrap confidence intervals, NOT standard error. Use `errorbar='se'` or `errorbar='sd'` if you intend standard error or standard deviation.
- **Forgetting inner='quartile' on violins**: Without `inner='quartile'`, the interior of the violin defaults to a miniature box that can be difficult to interpret; quartile lines clearly delineate $Q_1, Q_2, Q_3$.

## 4C.3 Faceting Across Conditions with catplot & FacetGrid

### 4.3.1 Theory & Biological Context

Biological experiments frequently investigate factorial designs: Multiple genotypes $\times$ Multiple treatments $\times$ Multiple timepoints.

Seaborn enables multidimensional decomposition via figure-level functions:
- **`sns.catplot()`**: For categorical variables (box, violin, bar, strip) across `col` and `row` subplots.
- **`sns.relplot()`**: For relational plots (scatter, line).
- **`sns.FacetGrid`**: Custom canvas grid for arbitrary plotting functions.

### 4.3.2 Worked Example

In [ ]:
# 4C.3 Worked Example: Faceting Expression Across Replicates and Conditions
# Let's inspect df_tidy with condition and replicate
g = sns.catplot(
    data=df_tidy[df_tidy["gene"].isin(["TP53", "BRCA1", "EGFR", "MYC"])],
    x="gene", y="expression", hue="condition",
    kind="bar", palette=["#4c72b0", "#c44e52"],
    height=3.5, aspect=1.3, errorbar="se", capsize=0.1
)

g.set_axis_labels("Target Oncogene / Tumor Suppressor", "Expression (Normalized TPM)")
g.fig.suptitle("Gene Expression Grouped by Experimental Condition", y=1.03, fontweight="bold")
plt.show()

### 4.3.3 ✏️ Try It: Tissue Replicate Faceter

Write a function `facet_tissue_replicates(df: pd.DataFrame) -> sns.FacetGrid`:
- Create a `sns.catplot` using `data=df`, `x="tissue"`, `y="tpm"`, `col="replicate"`.
- Use `kind="bar"`, `palette="Blues_d"`.
- Rotate x tick labels on each axis by 45 degrees.
- Return the `FacetGrid`.

In [ ]:
def facet_tissue_replicates(df: pd.DataFrame) -> sns.FacetGrid:
    # TODO: your code here
    ...

In [ ]:
# ✅ Test — run me. No output means you passed.
grid = facet_tissue_replicates(tissues_df)
assert isinstance(grid, sns.FacetGrid)
plt.close(grid.fig)
print("Section 4C.3 passed ✔")

### 4.3.4 ⚠️ Common Mistakes

- **Modifying FacetGrid using plt.title()**: Figure-level functions (`catplot`, `relplot`) return a `FacetGrid`, not an `Axes`. Modifying them requires `g.fig.suptitle()` or iterating over `g.axes`.
- **Inconsistent y-axis scales across facets**: By default `sharey=True` is enabled; disabling it (`sharey=False`) should only be done with care, as it can mislead comparisons across facets.

## 4C.4 Palettes & Colorblind Safety: Qualitative, Sequential, Diverging

### 4.4.1 Theory & Biological Context

Color is an informational channel, not mere decoration. In scientific publishing, selecting an inappropriate color scheme damages accessibility and introduces perceptual artifacts.

### Three Classes of Color Palettes:
1. **Qualitative Palettes**: For unordered nominal categories (e.g. Tissues, Genotypes).
   - Examples: `colorblind`, `Set2`, `tab10`.
   - Goal: Maximize perceptual distance between classes while avoiding bright/dim bias.
2. **Sequential Palettes**: For continuous quantities with an ordered progression from low to high (e.g. Read counts, Gene length).
   - Examples: `viridis`, `plasma`, `mako`, `Blues`.
   - Goal: Monotonically increasing perceptual luminance.
3. **Diverging Palettes**: For quantities with a meaningful zero or neutral center point (e.g. $\log_2$ Fold Change, correlation $r$, z-score).
   - Examples: `coolwarm`, `vlag`, `icefire`.
   - Goal: Equal visual weight given to positive and negative deviations from zero.

### 4.4.2 Worked Example

In [ ]:
# 4C.4 Worked Example: Inspecting Palettes for Publication
fig, axes = plt.subplots(3, 1, figsize=(8, 3.5), dpi=150)

palette_specs = [
    ("colorblind", "Qualitative (Nominal Categories)"),
    ("mako", "Sequential (Continuous Abundance)"),
    ("vlag", "Diverging (Fold Change centered at 0)")
]

for idx, (p_name, p_desc) in enumerate(palette_specs):
    pal = sns.color_palette(p_name, 8)
    for j, c in enumerate(pal):
        axes[idx].add_patch(plt.Rectangle((j, 0), 1, 1, color=c))
    axes[idx].set_xlim(0, 8)
    axes[idx].set_ylim(0, 1)
    axes[idx].axis("off")
    axes[idx].set_title(f"{p_desc}: '{p_name}'", fontsize=9, fontweight="bold", loc="left")

plt.tight_layout()
plt.show()

### 4.4.3 ✏️ Try It: Data-Driven Palette Selector

Write a function `get_palette_for_data(data_type: str, n_colors: int = 5) -> list`:
- If `data_type == "categorical"` -> return `sns.color_palette("colorblind", n_colors)`.
- If `data_type == "sequential"` -> return `sns.color_palette("viridis", n_colors)`.
- If `data_type == "diverging"` -> return `sns.color_palette("coolwarm", n_colors)`.
- Otherwise raise `ValueError("Unknown data type")`.

In [ ]:
def get_palette_for_data(data_type: str, n_colors: int = 5) -> list:
    # TODO: your code here
    ...

In [ ]:
# ✅ Test — run me. No output means you passed.
pal_cat = get_palette_for_data("categorical", 4)
assert len(pal_cat) == 4
pal_div = get_palette_for_data("diverging", 6)
assert len(pal_div) == 6
print("Section 4C.4 passed ✔")

### 4.4.4 ⚠️ Common Mistakes

- **Using diverging palettes on strictly positive data**: Applying `coolwarm` or `vlag` to raw TPM counts (> 0) falsely implies a meaningful zero center point.
- **Using red/green palettes for binary phenotypes**: Fails accessibility checks for deuteranopia/protanopia color vision deficiencies.

## 4C.5 Themes & Publication Styling: Context, Ticks, Despining

### 4.5.1 Theory & Biological Context

Publishing scientific figures requires clean typography, high contrast, and the elimination of extraneous graphical elements (Tufte's 'data-to-ink ratio').

### Seaborn Theming System:
- **`sns.set_theme()`**:
  - `style`: `'ticks'`, `'white'`, `'whitegrid'`, `'darkgrid'`. For papers, `'ticks'` or `'white'` is universally preferred.
  - `context`: `'paper'`, `'notebook'`, `'talk'`, `'poster'`. Scales line widths and fonts.
  - `font_scale`: Proportional multiplier (e.g. `1.1` or `1.2`).
- **`sns.despine()`**: Removes top and right spines, optionally offsetting axes or trimming to data ranges (`trim=True`).

### 4.5.2 Worked Example

In [ ]:
# 4C.5 Worked Example: Applying Publication Styling
sns.set_theme(style="ticks", context="paper", font_scale=1.1)

fig, ax = plt.subplots(figsize=(6, 4), dpi=150)
sns.barplot(data=tissues_df, x="tissue", y="tpm", ax=ax, palette="colorblind", errorbar="se", capsize=0.1)
ax.set_title("Expression Across Tissues (Publication Theme)", fontweight="bold")
ax.set_ylabel("Normalized TPM")
ax.set_xlabel("Organ Tissue")
ax.tick_params(axis="x", rotation=30)

# Despine top and right, with axis offset
sns.despine(ax=ax, offset=5, trim=True)

plt.tight_layout()
fig.savefig(OUT_DIR / "4C5_publication_styled.png", dpi=300)
plt.show()

### 4.5.3 ✏️ Try It: Publication Theme Applier

Write a function `apply_paper_theme() -> None`:
- Set seaborn theme with `style="ticks"`, `context="paper"`, and `font_scale=1.15`.
- Set Matplotlib's default DPI to 150 (`plt.rcParams['figure.dpi'] = 150`).

In [ ]:
def apply_paper_theme() -> None:
    # TODO: your code here
    ...

In [ ]:
# ✅ Test — run me. No output means you passed.
apply_paper_theme()
assert plt.rcParams["figure.dpi"] == 150
print("Section 4C.5 passed ✔")

### 4.5.4 ⚠️ Common Mistakes

- **Submitting dark background figures to journals**: `darkgrid` or dark mode styles consume enormous ink on printed pages and look unprofessional in formal manuscript PDF layouts.
- **Tiny unreadable tick labels in multi-panel figures**: Always use `context='paper'` with appropriate `font_scale` so figures remain legible when scaled down to column widths (85 mm or 170 mm).

## 4C.6 Biology Showcase: Volcano Plot & Clustered Heatmap

### 4.6.1 Theory & Biological Context

To synthesize our visual skills, we construct two signature figures of computational biology:

### 1. The Volcano Plot:
- **X-axis**: $\log_2(\text{Fold Change})$ — biological effect magnitude.
- **Y-axis**: $-\log_{10}(p\text{-value})$ — statistical confidence.
- Points in the upper-right quadrant: Significantly Upregulated.
- Points in the upper-left quadrant: Significantly Downregulated.
- Standard significance cutoffs: $|\log_2 \text{FC}| \ge 1.0$ (2-fold change) and adjusted $p \le 0.05$ ($-\log_{10} p \ge 1.30$).
- Key biological feature: Labeling top driver genes with arrow annotations.

### 2. The Clustered Heatmap (`sns.clustermap`):
- Performs agglomerative hierarchical clustering with Euclidean distance and Ward or Average linkage.
- Standardizes rows using z-scores ($z = \frac{x - \mu}{\sigma}$) to compare expression profiles across genes with vastly different baseline expression.

### 4.6.2 Worked Example

In [ ]:
# 4C.6 Worked Example: Volcano Plot with Annotated Driver Genes
de_path = DATA_DIR / "tables" / "differential_expression.csv"
df_de = pd.read_csv(de_path).copy()

# Compute -log10(pvalue)
df_de["neg_log10_p"] = -np.log10(df_de["pvalue"])

# Categorize significance
fc_cutoff = 1.0
pval_cutoff = 0.05
neg_log_p_cutoff = -np.log10(pval_cutoff)

def assign_status(row):
    if row["padj"] < pval_cutoff and row["log2FoldChange"] >= fc_cutoff:
        return "Upregulated"
    elif row["padj"] < pval_cutoff and row["log2FoldChange"] <= -fc_cutoff:
        return "Downregulated"
    return "Not Significant"

df_de["status"] = df_de.apply(assign_status, axis=1)

fig, ax = plt.subplots(figsize=(8, 5.5), dpi=150)
palette_map = {"Upregulated": "#d95f02", "Downregulated": "#7570b3", "Not Significant": "#cccccc"}

sns.scatterplot(
    data=df_de, x="log2FoldChange", y="neg_log10_p", hue="status",
    palette=palette_map, alpha=0.85, s=45, ax=ax, edgecolor="black", lw=0.3
)

# Threshold guideline lines
ax.axhline(neg_log_p_cutoff, ls="--", color="grey", lw=1.2, label=f"padj = {pval_cutoff}")
ax.axvline(fc_cutoff, ls=":", color="grey", lw=1.2)
ax.axvline(-fc_cutoff, ls=":", color="grey", lw=1.2)

# Annotate key genes: TP53, BRCA1 if present
for _, row in df_de.iterrows():
    if row["gene"] in ["TP53", "BRCA1"]:
        ax.annotate(
            row["gene"],
            xy=(row["log2FoldChange"], row["neg_log10_p"]),
            xytext=(row["log2FoldChange"] + 0.3, row["neg_log10_p"] + 0.5),
            arrowprops=dict(arrowstyle="->", color="black", lw=1.2),
            fontweight="bold", fontsize=9
        )

ax.set_title("Volcano Plot: Differential Gene Expression", fontsize=12, fontweight="bold")
ax.set_xlabel("log2(Fold Change: Treated / Control)", fontsize=10)
ax.set_ylabel("-log10(p-value)", fontsize=10)
ax.legend(title="Expression Status", frameon=True, loc="upper right")
sns.despine(ax=ax)

plt.tight_layout()
fig.savefig(OUT_DIR / "4C6_volcano_showcase.png", dpi=300)
plt.show()

### 4.6.3 ✏️ Try It: Hierarchical Clustermap Builder

Write a function `build_clustermap(df_matrix: pd.DataFrame, out_path: Path) -> sns.matrix.ClusterGrid`:
- Take `df_matrix` (from `expression_matrix.csv`), set `'gene'` as index.
- Compute row-wise z-scores (`z_score=0`).
- Use `cmap="vlag"`, `figsize=(8, 7)`.
- Save the clustermap to `out_path` with `dpi=300`.
- Return the `ClusterGrid` object.

In [ ]:
def build_clustermap(df_matrix: pd.DataFrame, out_path: Path) -> sns.matrix.ClusterGrid:
    # TODO: your code here
    ...

In [ ]:
# ✅ Test — run me. No output means you passed.
out_cm = OUT_DIR / "test_clustermap.png"
cg = build_clustermap(df_wide, out_cm)
assert out_cm.exists()
plt.close(cg.fig)
print("Section 4C.6 passed ✔")

### 4.6.4 ⚠️ Common Mistakes

- **Omitting z_score=0 in clustered heatmaps**: Without row z-score standardization, highly expressed housekeeping genes dominate the color scale, completely masking subtle differential expression in transcription factors.
- **Confusing p-value with adjusted p-value (padj)**: In high-throughput omics testing thousands of hypotheses simultaneously, raw p-values incur high false discovery rates; always filter significance by FDR / Benjamini-Hochberg `padj`.

### Practice Exercise 4C.1 (★) — Tidy Expression Transformation & Styled Boxplot

**Problem**: Write a function `plot_tidy_expression_boxplot(csv_path: Path, out_path: Path) -> Path`:
- Load `expression_tissues.csv`.
- Create a boxplot using `sns.boxplot(data=df, x="tissue", y="tpm", palette="Set2", width=0.5)`.
- Overlay jittered individual data points using `sns.stripplot(data=df, x="tissue", y="tpm", color="black", size=4, alpha=0.6, jitter=0.2)`.
- Set title: `"Tissue-Specific Expression (Box + Jitter)"`.
- Despine top and right borders.
- Save to `out_path` with `dpi=300, bbox_inches='tight'`.
- Return `out_path`.


- **Input**: Path to expression_tissues.csv and target image Path
- **Expected Output**: Path to saved figure

In [ ]:
def plot_tidy_expression_boxplot(csv_path: Path, out_path: Path) -> Path:
    # TODO: your code here
    ...

In [ ]:
# ✅ Test — run me. No output means you passed.
out_box_jitter = OUT_DIR / "tidy_boxplot_jitter.png"
p_box = plot_tidy_expression_boxplot(DATA_DIR / "tables" / "expression_tissues.csv", out_box_jitter)
assert p_box.exists() and p_box.stat().st_size > 15000
print("Ex 4C.1 passed ✔")

### Practice Exercise 4C.2 (★★) — Condition-Annotated Clustered Heatmap

**Problem**: Write a function `generate_annotated_clustermap(matrix_csv: Path, out_path: Path) -> Path`:
- Load `expression_matrix.csv`, set `'gene'` as index.
- Subset the first 15 genes.
- Create column color mapping for conditions: `'Control'` samples in `"#4c72b0"`, `'Treated'` samples in `"#d95f02"`.
- Generate `sns.clustermap` with `z_score=0`, `cmap="vlag"`, `col_colors=col_colors`, `figsize=(8, 7)`.
- Save figure to `out_path` with `dpi=300`.
- Return `out_path`.


- **Input**: Path to expression_matrix.csv and output image Path
- **Expected Output**: Path to saved clustered heatmap image

In [ ]:
def generate_annotated_clustermap(matrix_csv: Path, out_path: Path) -> Path:
    # TODO: your code here
    ...

In [ ]:
# ✅ Test — run me. No output means you passed.
out_annot_cm = OUT_DIR / "annotated_clustermap.png"
p_cm = generate_annotated_clustermap(DATA_DIR / "tables" / "expression_matrix.csv", out_annot_cm)
assert p_cm.exists() and p_cm.stat().st_size > 20000
print("Ex 4C.2 passed ✔")

### Practice Exercise 4C.3 (★★★) — Publication-Ready Volcano Plot with Highlighted Driver Genes

**Problem**: Write a function `generate_publication_volcano(diff_csv: Path, out_path: Path) -> Path`:
- Load `differential_expression.csv`.
- Calculate `neg_log10_p = -np.log10(df["pvalue"])`.
- Classify genes into three groups:
  - `"Upregulated"`: `padj < 0.05` and `log2FoldChange >= 1.0` (Color: `"#d95f02"`)
  - `"Downregulated"`: `padj < 0.05` and `log2FoldChange <= -1.0` (Color: `"#7570b3"`)
  - `"Not Significant"`: All other genes (Color: `"#bdbdbd"`)
- Plot with `sns.scatterplot(..., alpha=0.8, s=50)`.
- Draw dashed grey lines for `padj = 0.05` ($-\log_{10}(0.05) \approx 1.301$) and $|\log_2 \text{FC}| = 1.0$.
- Annotate the top 2 genes with highest `neg_log10_p` with their gene symbols and arrow annotations.
- Add formatted title, labels, legend, and despine.
- Save to `out_path` with `dpi=300, bbox_inches='tight'`.
- Return `out_path`.


- **Input**: Path to differential_expression.csv and target image Path
- **Expected Output**: Path to saved Volcano plot image

In [ ]:
def generate_publication_volcano(diff_csv: Path, out_path: Path) -> Path:
    # TODO: your code here
    ...

In [ ]:
# ✅ Test — run me. No output means you passed.
out_volcano = OUT_DIR / "volcano_final.png"
p_vol = generate_publication_volcano(DATA_DIR / "tables" / "differential_expression.csv", out_volcano)
assert p_vol.exists() and p_vol.stat().st_size > 20000
print("Ex 4C.3 passed ✔")

## 6. Summary & Cheat Sheet

Seaborn elevates Python data visualization by operating directly on tidy DataFrames. By automating statistical aggregation, error bar estimation, and multidimensional faceting (`hue`, `col`, `row`), it streamlines the production of publication-ready genomic figures.

### Quick Reference Table

| Concept / Operation | Syntax / Function | Biological Application | Key Parameter / Note |
| --- | --- | --- | --- |
| Tidy Reshape | pd.melt(df, id_vars=[...], ...) | Wide matrix to long format | Mandatory for Seaborn faceting |
| Bar Plot | sns.barplot(data, x, y, hue) | Grouped means with CI/SE | Set errorbar='se' for standard error |
| Distribution | sns.histplot(data, x, kde=True) | Distribution with KDE curve | Smooths discrete bins |
| Split Violin | sns.violinplot(..., split=True) | Pairwise group comparisons | Reveals multimodal phenotypes |
| Faceting | sns.catplot(..., col='replicate') | Multi-factor grid exploration | Returns FacetGrid |
| Clustered Heatmap | sns.clustermap(mat, z_score=0) | Unsupervised bi-clustering | Always standardize row z-scores |
| Volcano Plot | sns.scatterplot(log2FC vs -log10 p) | Differential expression showcase | Annotate driver genes |

## 7. Further Reading & Resources

- [Seaborn Official Gallery & Tutorial](https://seaborn.pydata.org/tutorial.html): Official user guide and comprehensive example gallery.
- [Tidy Data by Hadley Wickham](https://vita.had.co.nz/papers/tidy-data.pdf): The foundational paper defining tidy data structures for statistical computing.
- [Nature Methods: Volcano Plots in Omics](https://www.nature.com/articles/nmeth.3090): Best practices for plotting and thresholding high-throughput differential assays.